# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform initial processing on the FAIR² dataset using the `mlcroissant` library. All references to schemas, record sets, fields, and columns are made via their `@id` properties to ensure reproducibility and correctness.

### Dataset Source
The dataset is published as a Croissant JSON-LD schema and is accessed via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get and print basic metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}\n")
if hasattr(metadata, 'license'):
    print(f"License: {metadata.license}\n")

## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers.

A Record Set is an entity (such as a logical sheet or table) as defined by the Croissant specification. Record sets may contain fields which correspond to data columns.

In [ ]:
print("Available Record Sets and Fields (@id):\n")
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id', str(field))
            else:
                field_id = str(field)
            print(f"  - Field @id: {field_id}")

In [ ]:
# If at least one record set exists, preview a few records using its @id (else, leave empty)
if len(record_sets) > 0:
    record_set_id = record_sets[0]['@id']
    print(f"\nPreview first 3 records from RecordSet @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        if i >= 3:
            break
        print(record)

## 3. Data Extraction

Load data from all available record sets into pandas DataFrames for analysis.
All access is via `@id` fields as prescribed by the Croissant specification.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"Loaded {df.shape[0]} records and {df.shape[1]} fields from RecordSet @id: {rs_id}")

# Display columns of the first record set, if available
if len(record_set_ids) > 0 and not dataframes[record_set_ids[0]].empty:
    print(f"\nColumns in RecordSet @id {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply standard data processing steps such as filtering, normalization, and grouping.
Field selection is performed via `@id`.

In [ ]:
# Example EDA on the first available record set with numeric columns
import numpy as np

if len(record_set_ids) > 0 and not dataframes[record_set_ids[0]].empty:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Try to detect a numeric field by checking dtypes
    numeric_field_id = None
    for col in df.columns:
        # Only consider columns with numeric types
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field detected for EDA. Skipping this section.")
    else:
        print(f"Using numeric field for EDA: {numeric_field_id}\n")
        threshold = df[numeric_field_id].mean()  # Use mean as threshold for illustration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (the first non-numeric field if available)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization

Visualize the distribution of a selected numeric field and any relationship to a (potential) grouping field. Both axes and labels are indicated via field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if len(record_set_ids) > 0 and not dataframes[record_set_ids[0]].empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=25)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, visualize group means
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df.index, y=grouped_df['mean_' + numeric_field_id])
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook showcased the usage of the FAIR² dataset Croissant schema with the `mlcroissant` library. 

- We loaded metadata and explored the logical table/recordset structure using `@id` references.
- Extracted tabular data directly into pandas for convenient analysis.
- Performed example EDA: filtering, normalization, and basic groupby aggregation, always referencing fields by Croissant `@id`.
- Visualized distributions and aggregated statistics.

Further analysis can build on this scaffold to address specific research or policy questions using the provided fields and record sets.